# GYAFC train-set exploration for activation steering

This notebook uses only the aligned training pairs from the two GYAFC domains:

- `Entertainment_Music/train/{informal,formal}`
- `Family_Relationships/train/{informal,formal}`

The immediate question is whether these pairs contain a repeatable *formalization signal* worth looking for inside a language model. We will:

1. verify that the files are truly aligned;
2. describe what changes from informal to formal;
3. inspect likely noise and semantic drift;
4. estimate a paired direction in TF-IDF text space and test it on held-out pairs;
5. test whether a direction learned in one domain transfers to the other.

> **Important limitation:** a TF-IDF direction is not a model activation direction. It is a cheap feasibility check for a consistent, approximately linear signal in the dataset. Establishing an activation direction requires extracting hidden states from a chosen model.

In [1]:
from pathlib import Path
from difflib import SequenceMatcher
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 120)
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 1. Locate and load the aligned training pairs

Change `DATA_ROOT` if the corpus moves. Each informal line must correspond to the formal line at the same index.

In [2]:
DATA_ROOT = Path("/Users/nandanprince/Downloads/GYAFC_Corpus")

DOMAINS = {
    "entertainment_music": "Entertainment_Music",
    "family_relationships": "Family_Relationships",
}

assert DATA_ROOT.is_dir(), f"GYAFC directory not found: {DATA_ROOT}"
DATA_ROOT

PosixPath('/Users/nandanprince/Downloads/GYAFC_Corpus')

In [3]:
def read_lines(path: Path) -> list[str]:
    with path.open(encoding="utf-8") as handle:
        return [line.rstrip("\n\r") for line in handle]


def load_domain(domain_label: str, directory_name: str) -> pd.DataFrame:
    train_dir = DATA_ROOT / directory_name / "train"
    informal = read_lines(train_dir / "informal")
    formal = read_lines(train_dir / "formal")

    assert len(informal) == len(formal), (
        f"Unaligned files for {domain_label}: "
        f"{len(informal):,} informal vs {len(formal):,} formal"
    )

    return pd.DataFrame(
        {
            "domain": domain_label,
            "domain_row": np.arange(len(informal)),
            "informal": informal,
            "formal": formal,
        }
    )


pairs = pd.concat(
    [load_domain(label, directory) for label, directory in DOMAINS.items()],
    ignore_index=True,
)
pairs.insert(0, "pair_id", np.arange(len(pairs)))

print(f"Loaded {len(pairs):,} aligned train pairs.")
pairs.groupby("domain").size().rename("pairs").to_frame()

Loaded 104,562 aligned train pairs.


,pairs
domain,
entertainment_music,52595
family_relationships,51967


## 2. Integrity and duplication checks

Exact copies are not necessarily errors: they indicate examples for which no rewrite was made. But many copies or duplicate pairs can dominate a mean difference vector, so quantify them before extraction.

In [4]:
def normalized_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())


pairs["informal_norm"] = pairs["informal"].map(normalized_text)
pairs["formal_norm"] = pairs["formal"].map(normalized_text)
pairs["exact_copy"] = pairs["informal_norm"] == pairs["formal_norm"]
pairs["blank_informal"] = pairs["informal_norm"].eq("")
pairs["blank_formal"] = pairs["formal_norm"].eq("")
pairs["duplicate_pair"] = pairs.duplicated(
    ["informal_norm", "formal_norm"], keep=False
)

integrity = pairs.groupby("domain").agg(
    pairs=("pair_id", "size"),
    blank_informal=("blank_informal", "sum"),
    blank_formal=("blank_formal", "sum"),
    exact_copies=("exact_copy", "sum"),
    rows_in_duplicated_pairs=("duplicate_pair", "sum"),
)
integrity["exact_copy_pct"] = 100 * integrity["exact_copies"] / integrity["pairs"]
integrity.round(2)

,pairs,blank_informal,blank_formal,exact_copies,rows_in_duplicated_pairs,exact_copy_pct
domain,,,,,,
entertainment_music,52595,0,0,511,163,0.97
family_relationships,51967,0,0,689,71,1.33


In [5]:
# A reproducible first look at pairs from both domains.
pairs.groupby("domain", group_keys=False).sample(
    n=8, random_state=RANDOM_SEED
)[["pair_id", "domain", "informal", "formal"]]

,pair_id,domain,informal,formal
35747,35747,entertainment_music,"oh,N it might soud weird,but I woul also choose Skandar Keynes he's( or used 2 B) such a cutie!!!!","This may sound strange but I would also choose Skandar Keynes. He is, or at least used to be, very attractive."
40043,40043,entertainment_music,are they really heavy and whats the best album to get?,"Are they really heavy? Also, what is the best album to get?"
462,462,entertainment_music,I pod's - you can watch podcasts or little movies on the screen.,Ipods are devices that allow you to watch podcasts or movies on its screen.
48372,48372,entertainment_music,my momma was wondering the same thing,My mother was thinking about the same thing.
25495,25495,entertainment_music,"then it emailed me the old info, when prompted.",It was set to email me the old information when prompted.
33903,33903,entertainment_music,"Anyway, once you get a building and set up your projection, etc.","As soon as you get a building and can set up your projection, etc."
48794,48794,entertainment_music,none scares hte shit out of me for too long.. the most 2 days.,None of them frighten me for very long. The maximum is 2 days.
24666,24666,entertainment_music,The tale of two cities !,The book is The Tale of Two Cities.
63373,63373,family_relationships,But give me credit for trying.,Please give me credit for trying. I did my best.
79475,79475,family_relationships,Same reson why people hate gay folks.,It is the same reason why some people dislike homosexuals.


## 3. What changes during formalization?

These surface measurements do not define formality. They reveal recurring correlates and possible confounds that an activation vector could learn: length, contractions, slang, casing, punctuation, or rewriting intensity.

In [7]:
WORD_RE = re.compile(r"\b[\w']+\b")
CONTRACTION_RE = re.compile(
    r"\b(?:ain't|aren't|can't|couldn't|didn't|doesn't|don't|hadn't|hasn't|haven't|"
    r"he's|i'd|i'll|i'm|i've|isn't|it's|shouldn't|that's|they're|wasn't|weren't|"
    r"what's|won't|wouldn't|you're|you've)\b",
    flags=re.IGNORECASE,
)
SLANG_RE = re.compile(
    r"\b(?:u|ur|r|lol|omg|idk|imo|btw|thx|thanx|wanna|gonna|gotta|cuz|coz|kinda)\b",
    flags=re.IGNORECASE,
)
POLITE_RE = re.compile(r"\b(?:please|thank you|thanks|appreciate|kindly)\b", re.IGNORECASE)


def surface_features(text: str) -> dict[str, float]:
    words = WORD_RE.findall(text)
    letters = [char for char in text if char.isalpha()]
    return {
        "characters": len(text),
        "words": len(words),
        "mean_word_length": np.mean([len(word) for word in words]) if words else 0.0,
        "contractions": len(CONTRACTION_RE.findall(text)),
        "slang_tokens": len(SLANG_RE.findall(text)),
        "politeness_terms": len(POLITE_RE.findall(text)),
        "exclamation_marks": text.count("!"),
        "question_marks": text.count("?"),
        "uppercase_share": (
            sum(char.isupper() for char in letters) / len(letters) if letters else 0.0
        ),
        "ends_with_terminal_punctuation": float(text.rstrip().endswith((".", "!", "?"))),
    }


feature_names = list(surface_features("Example.").keys())

# Make this cell safe to rerun: remove columns accidentally duplicated by an
# earlier execution, then replace feature columns instead of concatenating them.
pairs = pairs.loc[:, ~pairs.columns.duplicated()].copy()
for style in ("informal", "formal"):
    feature_frame = pd.DataFrame(pairs[style].map(surface_features).tolist())
    for name in feature_names:
        pairs[f"{style}_{name}"] = feature_frame[name].to_numpy()

assert not pairs.columns.duplicated().any(), "Duplicate columns detected."

In [8]:
# Mean feature values by domain and style.
feature_rows = []
for domain, group in pairs.groupby("domain"):
    for style in ("informal", "formal"):
        row = {"domain": domain, "style": style}
        row.update(
            {name: group[f"{style}_{name}"].mean() for name in feature_names}
        )
        feature_rows.append(row)

feature_summary = pd.DataFrame(feature_rows).set_index(["domain", "style"])
feature_summary.round(3)

characters  \
domain               style                                                                                        
entertainment_music  informal  informal_characters    50.429109
informal_characters    50.429109
dtype: float64   
                     formal        formal_characters    54.528434
formal_characters    54.528434
dtype: float64   
family_relationships informal  informal_characters    52.271076
informal_characters    52.271076
dtype: float64   
                     formal          formal_characters    57.23207
formal_characters    57.23207
dtype: float64   

                                                                                                words  \
domain               style                                                                              
entertainment_music  informal  informal_words    10.065691
informal_words    10.065691
dtype: float64   
                     formal        formal_words    10.708813
formal_words    10.708813
dtype: float64   
family_relationships informal  informal_words    10.734293
informal_words    10.734293
dtype: float64   
                     formal          formal_words    11.34239
formal_words    11.34239
dtype: float64   

                                                                                                         mean_word_length  \
domain               style                                                                                                  
entertainment_music  informal  informal_mean_word_length    3.879278
informal_mean_word_length    3.879278
dtype: float64   
                     formal        formal_mean_word_length    4.026845
formal_mean_word_length    4.026845
dtype: float64   
family_relationships informal    informal_mean_word_length    3.79011
informal_mean_word_length    3.79011
dtype: float64   
                     formal        formal_mean_word_length    4.006267
formal_mean_word_length    4.006267
dtype: float64   

                                                                                                     contractions  \
domain               style                                                                                          
entertainment_music  informal  informal_contractions    0.199277
informal_contractions    0.199277
dtype: float64   
                     formal        formal_contractions    0.106341
formal_contractions    0.106341
dtype: float64   
family_relationships informal  informal_contractions    0.219216
informal_contractions    0.219216
dtype: float64   
                     formal        formal_contractions    0.101218
formal_contractions    0.101218
dtype: float64   

                                                                                                     slang_tokens  \
domain               style                                                                                          
entertainment_music  informal  informal_slang_tokens    0.115087
informal_slang_tokens    0.115087
dtype: float64   
                     formal        formal_slang_tokens    0.006198
formal_slang_tokens    0.006198
dtype: float64   
family_relationships informal    informal_slang_tokens    0.15964
informal_slang_tokens    0.15964
dtype: float64   
                     formal        formal_slang_tokens    0.000808
formal_slang_tokens    0.000808
dtype: float64   

                                                                                                         politeness_terms  \
domain               style                                                                                                  
entertainment_music  informal  informal_politeness_terms    0.007149
informal_politeness_terms    0.007149
dtype: float64   
                     formal        formal_politeness_terms    0.019907
formal_politeness_terms    0.019907
dtype: float64   
family_relationships informal  informal_politeness_terms    0.007428
informal_politeness_terms    0.007428
dtype: float64 

In [9]:
# Paired changes are more informative than independent style averages.
paired_change_rows = []
for domain, group in pairs.groupby("domain"):
    for name in feature_names:
        delta = group[f"formal_{name}"] - group[f"informal_{name}"]
        paired_change_rows.append(
            {
                "domain": domain,
                "feature": name,
                "mean_formal_minus_informal": delta.mean(),
                "median_formal_minus_informal": delta.median(),
                "pct_increased": 100 * (delta > 0).mean(),
                "pct_decreased": 100 * (delta < 0).mean(),
            }
        )

paired_changes = pd.DataFrame(paired_change_rows)
paired_changes.round(3)

,domain,feature,mean_formal_minus_informal,median_formal_minus_informal,pct_increased,pct_decreased
0,entertainment_music,characters,formal_characters NaN formal_characters NaN informal_characters NaN informal_characters NaN dtype: float64,formal_characters NaN formal_characters NaN informal_characters NaN informal_characters NaN dtype: float64,formal_characters 0.0 formal_characters 0.0 informal_characters 0.0 informal_characters 0.0 dtype: f...,formal_characters 0.0 formal_characters 0.0 informal_characters 0.0 informal_characters 0.0 dtype: f...
1,entertainment_music,words,formal_words NaN formal_words NaN informal_words NaN informal_words NaN dtype: float64,formal_words NaN formal_words NaN informal_words NaN informal_words NaN dtype: float64,formal_words 0.0 formal_words 0.0 informal_words 0.0 informal_words 0.0 dtype: float64,formal_words 0.0 formal_words 0.0 informal_words 0.0 informal_words 0.0 dtype: float64
2,entertainment_music,mean_word_length,formal_mean_word_length NaN formal_mean_word_length NaN informal_mean_word_length NaN informal_mean_word_l...,formal_mean_word_length NaN formal_mean_word_length NaN informal_mean_word_length NaN informal_mean_word_l...,formal_mean_word_length 0.0 formal_mean_word_length 0.0 informal_mean_word_length 0.0 informal_mean_wor...,formal_mean_word_length 0.0 formal_mean_word_length 0.0 informal_mean_word_length 0.0 informal_mean_wor...
3,entertainment_music,contractions,formal_contractions NaN formal_contractions NaN informal_contractions NaN informal_contractions NaN dtyp...,formal_contractions NaN formal_contractions NaN informal_contractions NaN informal_contractions NaN dtyp...,formal_contractions 0.0 formal_contractions 0.0 informal_contractions 0.0 informal_contractions 0.0 ...,formal_contractions 0.0 formal_contractions 0.0 informal_contractions 0.0 informal_contractions 0.0 ...
4,entertainment_music,slang_tokens,formal_slang_tokens NaN formal_slang_tokens NaN informal_slang_tokens NaN informal_slang_tokens NaN dtyp...,formal_slang_tokens NaN formal_slang_tokens NaN informal_slang_tokens NaN informal_slang_tokens NaN dtyp...,formal_slang_tokens 0.0 formal_slang_tokens 0.0 informal_slang_tokens 0.0 informal_slang_tokens 0.0 ...,formal_slang_tokens 0.0 formal_slang_tokens 0.0 informal_slang_tokens 0.0 informal_slang_tokens 0.0 ...
5,entertainment_music,politeness_terms,formal_politeness_terms NaN formal_politeness_terms NaN informal_politeness_terms NaN informal_politeness_...,formal_politeness_terms NaN formal_politeness_terms NaN informal_politeness_terms NaN informal_politeness_...,formal_politeness_terms 0.0 formal_politeness_terms 0.0 informal_politeness_terms 0.0 informal_politene...,formal_politeness_terms 0.0 formal_politeness_terms 0.0 informal_politeness_terms 0.0 informal_politene...
6,entertainment_music,exclamation_marks,formal_exclamation_marks NaN formal_exclamation_marks NaN informal_exclamation_marks NaN informal_exclamat...,formal_exclamation_marks NaN formal_exclamation_marks NaN informal_exclamation_marks NaN informal_exclamat...,formal_exclamation_marks 0.0 formal_exclamation_marks 0.0 informal_exclamation_marks 0.0 informal_excla...,formal_exclamation_marks 0.0 formal_exclamation_marks 0.0 informal_exclamation_marks 0.0 informal_excla...
7,entertainment_music,question_marks,formal_question_marks NaN formal_question_marks NaN informal_question_marks NaN informal_question_marks ...,formal_question_marks NaN formal_question_marks NaN informal_question_marks NaN informal_question_marks ...,formal_question_marks 0.0 formal_question_marks 0.0 informal_question_marks 0.0 informal_question_marks...,formal_question_marks 0.0 formal_question_marks 0.0 informal_question_marks 0.0 informal_question_marks...
8,entertainment_music,uppercase_share,formal_uppercase_share NaN formal_uppercase_share NaN informal_uppercase_share NaN informal_uppercase_shar...,formal_uppercase_share NaN formal_uppercase_share NaN informal_uppercase_share NaN informal_uppercase_shar...,formal

### Transformation prevalence

A good extraction set should contain varied examples rather than being dominated by one shortcut. The table below estimates how often a few recognizable transformations occur. These are heuristic indicators, so use them for diagnosis rather than ground-truth labels.

In [12]:
transformation_flags = pd.DataFrame(
    {
        "contraction_removed": (
            pairs["informal_contractions"]
            > pairs["formal_contractions"]
        ),
        "slang_removed": (
            pairs["informal_slang_tokens"]
            > pairs["formal_slang_tokens"]
        ),
        "politeness_added": (
            pairs["formal_politeness_terms"]
            > pairs["informal_politeness_terms"]
        ),
        "terminal_punctuation_added": (
            pairs["formal_ends_with_terminal_punctuation"]
            > pairs["informal_ends_with_terminal_punctuation"]
        ),
        "word_count_increased": (
            pairs["formal_words"] > pairs["informal_words"]
        ),
        "word_count_decreased": (
            pairs["formal_words"] < pairs["informal_words"]
        ),
        "exact_copy": pairs["exact_copy"],
    }
)

transform_prevalence = pd.concat(
    [pairs[["domain"]], transformation_flags],
    axis=1,
)

(
    100
    * transform_prevalence.groupby("domain").mean()
).round(2).T.rename_axis("heuristic")

ValueError: Can only compare identically-labeled (both index and columns) DataFrame objects

## 4. Inspect possible noise and semantic drift

Character similarity is only a triage tool. Low similarity can mean a valuable structural rewrite, or it can mean the formal sentence changed the meaning. Extreme length ratios have the same ambiguity. Read these examples manually before deciding on filtering rules.

In [11]:
pairs["character_similarity"] = [
    SequenceMatcher(None, informal, formal, autojunk=False).ratio()
    for informal, formal in zip(pairs["informal_norm"], pairs["formal_norm"])
]
pairs["word_count_ratio"] = (pairs["formal_words"] + 1) / (pairs["informal_words"] + 1)

pairs.groupby("domain")[["character_similarity", "word_count_ratio"]].quantile(
    [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).round(3)

ValueError: Cannot set a DataFrame with multiple columns to the single column word_count_ratio

In [ ]:
# Review a deterministic mix of the least similar and most length-altered pairs.
review_ids = pd.Index(
    np.concatenate(
        [
            pairs.nsmallest(15, "character_similarity").index,
            pairs.nlargest(8, "word_count_ratio").index,
            pairs.nsmallest(8, "word_count_ratio").index,
        ]
    )
).unique()

pairs.loc[
    review_ids,
    [
        "pair_id",
        "domain",
        "character_similarity",
        "word_count_ratio",
        "informal",
        "formal",
    ],
].sort_values("character_similarity").round(3)

### Manual annotation sample

Read a balanced random sample and annotate the dominant changes outside this notebook (or add columns here). Useful labels include spelling/grammar, contraction expansion, slang removal, lexical substitution, syntactic restructuring, politeness, punctuation/casing, verbosity, exact copy, and probable meaning change. This manual step is important because surface heuristics cannot distinguish formalization from semantic drift.

In [ ]:
manual_review_sample = (
    pairs.loc[~pairs["exact_copy"]]
    .groupby("domain", group_keys=False)
    .sample(n=25, random_state=RANDOM_SEED)
    [["pair_id", "domain", "informal", "formal"]]
    .sort_values(["domain", "pair_id"])
)
manual_review_sample

## 5. Text-space paired-direction feasibility test

We now mirror the planned activation experiment using TF-IDF sentence vectors:

```text
delta_i = tfidf(formal_i) - tfidf(informal_i)
v_text  = mean(delta_i) on extraction pairs
```

The direction is estimated only on the extraction split. A held-out pair is correctly ordered when its formal sentence projects higher than its informal sentence. Pair IDs—not individual sentences—are split, which prevents the two sides of one pair from leaking across splits.

In [ ]:
extraction_ids, heldout_ids = train_test_split(
    pairs.index.to_numpy(),
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=pairs["domain"],
)

extraction = pairs.loc[extraction_ids].copy()
heldout = pairs.loc[heldout_ids].copy()

split_counts = pd.concat(
    [
        extraction.groupby("domain").size().rename("extraction"),
        heldout.groupby("domain").size().rename("heldout"),
    ],
    axis=1,
)
split_counts.loc["total"] = split_counts.sum()
split_counts

In [ ]:
# Fit the vocabulary/IDF on extraction sentences only.
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.98,
    max_features=30_000,
    sublinear_tf=True,
)
vectorizer.fit(pd.concat([extraction["informal"], extraction["formal"]]))

def style_matrices(frame: pd.DataFrame):
    informal_matrix = vectorizer.transform(frame["informal"])
    formal_matrix = vectorizer.transform(frame["formal"])
    return informal_matrix, formal_matrix


X_extract_informal, X_extract_formal = style_matrices(extraction)
X_heldout_informal, X_heldout_formal = style_matrices(heldout)

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out()):,}")
print(f"Extraction matrices: {X_extract_informal.shape}")
print(f"Held-out matrices:   {X_heldout_informal.shape}")

In [ ]:
def mean_paired_direction(informal_matrix, formal_matrix) -> np.ndarray:
    direction = np.asarray((formal_matrix - informal_matrix).mean(axis=0)).ravel()
    norm = np.linalg.norm(direction)
    if norm == 0:
        raise ValueError("The mean paired direction has zero norm.")
    return direction / norm


def evaluate_direction(informal_matrix, formal_matrix, direction: np.ndarray) -> dict[str, float]:
    differences = formal_matrix - informal_matrix
    paired_projection = np.asarray(differences @ direction).ravel()
    difference_norms = np.sqrt(np.asarray(differences.multiply(differences).sum(axis=1)).ravel())
    nonzero = difference_norms > 0
    cosine_alignment = paired_projection[nonzero] / difference_norms[nonzero]
    sd = paired_projection.std(ddof=1)
    return {
        "n_pairs": len(paired_projection),
        "pair_ordering_accuracy": np.mean(paired_projection > 0),
        "tie_rate": np.mean(np.isclose(paired_projection, 0.0)),
        "mean_projection_difference": paired_projection.mean(),
        "projection_effect_size": paired_projection.mean() / sd if sd > 0 else np.nan,
        "mean_delta_cosine_with_direction": cosine_alignment.mean(),
        "median_delta_cosine_with_direction": np.median(cosine_alignment),
    }


text_direction = mean_paired_direction(X_extract_informal, X_extract_formal)
pd.DataFrame(
    {
        "extraction": evaluate_direction(
            X_extract_informal, X_extract_formal, text_direction
        ),
        "heldout": evaluate_direction(
            X_heldout_informal, X_heldout_formal, text_direction
        ),
    }
).T.round(4)

### Interpret the diagnostic

- `pair_ordering_accuracy`: fraction of pairs for which the formal sentence scores higher; 0.50 is chance, while ties count as incorrect.
- `projection_effect_size`: mean paired projection divided by its standard deviation.
- `delta_cosine_with_direction`: whether individual pair differences point toward the mean direction. Sparse lexical deltas can make this modest even when ordering is useful.
- A large extraction/held-out gap suggests overfitting or unstable lexical shortcuts.

A result above chance supports trying hidden-state extraction. It does **not** show that a single activation direction exists or that adding such a direction will causally change generation.

In [ ]:
# Which words and phrases define the TF-IDF direction?
# This helps expose shortcuts such as spelling corrections, discourse markers, or domain terms.
feature_names_tfidf = vectorizer.get_feature_names_out()
top_k = 30
formal_indices = np.argsort(text_direction)[-top_k:][::-1]
informal_indices = np.argsort(text_direction)[:top_k]

top_direction_features = pd.DataFrame(
    {
        "formal_feature": feature_names_tfidf[formal_indices],
        "formal_weight": text_direction[formal_indices],
        "informal_feature": feature_names_tfidf[informal_indices],
        "informal_weight": text_direction[informal_indices],
    }
)
top_direction_features.round(5)

### Stability across independent extraction subsets

Two directions estimated from disjoint halves of the extraction data should be similar and should perform similarly on the same untouched held-out set. This is a stronger check than extraction accuracy alone.

In [ ]:
half_a_ids, half_b_ids = train_test_split(
    extraction.index.to_numpy(),
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=extraction["domain"],
)

half_directions = {}
for label, row_ids in {"half_a": half_a_ids, "half_b": half_b_ids}.items():
    half_informal, half_formal = style_matrices(extraction.loc[row_ids])
    half_directions[label] = mean_paired_direction(half_informal, half_formal)

half_direction_cosine = float(half_directions["half_a"] @ half_directions["half_b"])
half_results = pd.DataFrame(
    {
        label: evaluate_direction(
            X_heldout_informal, X_heldout_formal, direction
        )
        for label, direction in half_directions.items()
    }
).T

print(f"Cosine between directions from disjoint extraction halves: {half_direction_cosine:.4f}")
half_results.round(4)

## 6. Domain robustness

A direction that captures general formalization should transfer at least partially between Entertainment & Music and Family & Relationships. Strong within-domain results but chance cross-domain results would suggest topic-specific lexical shortcuts.

In [ ]:
def matrices_for_domain(frame: pd.DataFrame, domain: str):
    domain_frame = frame.loc[frame["domain"] == domain]
    informal_matrix, formal_matrix = style_matrices(domain_frame)
    return domain_frame, informal_matrix, formal_matrix


domain_directions = {}
for domain in DOMAINS:
    _, domain_informal, domain_formal = matrices_for_domain(extraction, domain)
    domain_directions[domain] = mean_paired_direction(domain_informal, domain_formal)

domain_results = []
for source_domain, direction in domain_directions.items():
    for target_domain in DOMAINS:
        target_frame, target_informal, target_formal = matrices_for_domain(
            heldout, target_domain
        )
        result = evaluate_direction(target_informal, target_formal, direction)
        domain_results.append(
            {
                "direction_learned_from": source_domain,
                "evaluated_on": target_domain,
                **result,
            }
        )

domain_direction_cosine = float(
    domain_directions["entertainment_music"]
    @ domain_directions["family_relationships"]
)
print(f"Cosine between domain-specific directions: {domain_direction_cosine:.4f}")
pd.DataFrame(domain_results).set_index(
    ["direction_learned_from", "evaluated_on"]
).round(4)

## 7. Decision checklist for the activation experiment

After running and reading the outputs, record answers to these questions:

1. **Pair quality:** How often do examples appear meaning-preserving? Which noise patterns need exclusion or stratification?
2. **Transformation mixture:** Is formalization mostly spelling and punctuation, or does it include consistent lexical and syntactic changes?
3. **Text-space coherence:** Is held-out pair-ordering accuracy clearly above 0.50? Are individual delta alignments consistently positive?
4. **Robustness:** Do directions transfer across both domains, and are the domain-specific directions similar?
5. **Shortcut risk:** Do the highest-weight features describe formality, or mostly topic, names, and annotation artifacts?

### Recommended extraction policy for the model stage

- Keep the split at the **pair level**.
- Use an extraction subset to estimate activation directions and a never-used held-out subset to test them.
- Report results separately for both domains as well as combined.
- Keep exact copies out of initial direction estimation because their deltas should be near zero; retain them as a useful control set.
- Do not automatically delete low-similarity pairs. Manually label a sample first, then define transparent filtering rules.
- In the next notebook, repeat the paired-direction analysis at every model layer using hidden-state vectors instead of TF-IDF vectors.

The dataset analysis can justify attempting activation extraction, but the decisive evidence will be held-out activation geometry: layer-wise pair ordering, cosine consistency, bootstrap stability, and cross-domain transfer.

## 8. Results and interpretation

### Dataset quality and the meaning of formalization

The two training domains contain **104,562 aligned pairs**: 52,595 from Entertainment & Music and 51,967 from Family & Relationships. No blank sentences were found. Exact copies account for only **0.97%** and **1.33%** of the two domains, so most pairs contain a non-zero transformation.

The rewrites show several recurring changes in both domains:

- Formal sentences are about **4.1–5.0 characters** and **0.61–0.64 words** longer on average.
- Word count increases in roughly **49%** of pairs and decreases in roughly **30%**, showing that formalization is not simply expansion.
- Contractions are removed in **13.5–15.4%** of pairs.
- The limited slang detector finds slang removal in **9.1–11.7%** of pairs.
- Terminal punctuation is added in **31.0%** of Entertainment & Music pairs and **17.7%** of Family & Relationships pairs.
- Exclamation marks, slang, contractions, and excessive capitalization decrease on average, while mean word length and complete punctuation increase.

Median character similarity is about **0.69** in both domains. The lowest 5% fall below approximately **0.35**, so a meaningful minority deserves manual review for semantic drift or annotation noise. These pairs should not be removed solely because they are lexically different: some may contain legitimate structural rewrites.

### Evidence for a shared text-space direction

A TF-IDF direction was estimated on 80% of the pairs using `mean(formal - informal)` and evaluated on the untouched 20%.

| Measurement | Result | Interpretation |
|---|---:|---|
| Extraction pair-ordering accuracy | 81.12% | The direction fits the examples used to estimate it. |
| Held-out pair-ordering accuracy | **80.83%** | The signal generalizes well beyond the extraction pairs; chance is 50%. |
| Held-out projection effect size | **0.794** | The average paired separation is substantial relative to its variation. |
| Cosine between directions from disjoint extraction halves | **0.986** | The aggregate direction is highly stable to the sampled pairs. |
| Cosine between domain-specific directions | **0.856** | The two domains share much of the same direction despite different topics. |
| Cross-domain pair-ordering accuracy | **78.5–79.6%** | A direction learned in one domain transfers to the other. |

The near-identical extraction and held-out accuracies argue against the result being simple sample overfitting. Cross-domain transfer also makes a purely topic-specific explanation less likely. The most influential TF-IDF features include contraction expansion and informal-token removal (`dont` → `do not`, `im` → `am`, `its` → `it is`, and removal of `ur` and `lol`). This confirms a real shared signal, but it also shows that the direction partly encodes visible lexical and orthographic shortcuts rather than a purely abstract concept of formality.

### Does this mean an activation direction exists?

**The results make an activation-direction experiment well motivated, but they do not prove that such a direction exists.** They establish three useful prerequisites:

1. Formal and informal sentences are systematically distinguishable.
2. The aggregate distinction is stable across independent samples.
3. Much of the distinction transfers across domains rather than depending only on topic.

There is also an important warning. The mean cosine between an individual TF-IDF pair difference and the aggregate direction is only about **0.043** (median **0.034**). Individual rewrites therefore do not all perform the same transformation. Formalization is a mixture of spelling correction, contraction expansion, slang removal, punctuation, lexical substitution, syntactic restructuring, politeness, and changes in explicitness. A model may compress these correlated properties into a useful shared activation direction, or it may represent them as several directions or a nonlinear subspace.

### Decision

Proceed to hidden-state extraction, but **do not proceed directly to activation steering**. For every model layer, estimate `mean(h_formal - h_informal)` on an extraction subset and repeat the held-out ordering, split-stability, and cross-domain tests used here. Compare mean-pooled and final-token representations. Keep exact copies as a control group, and report results both with and without manually identified noisy pairs. Only layers with stable held-out separation should be considered candidates for causal steering.